# ClearBank — Análise Financeira de Transações com Python

Este notebook processa o arquivo `transacoes.csv`, valida os dados, calcula métricas mensais, identifica transações suspeitas e exporta o resultado em `relatorio.json`.

---

### 📋 Instruções de Uso

1. Certifique-se de que o arquivo `transacoes.csv` está na mesma pasta deste notebook.
2. Execute as células **em ordem**, de cima para baixo.
3. A **Célula de Execução Principal** chama todas as funções e gera:
   - Relatório formatado no terminal (saída da célula)
   - Arquivo `relatorio.json` na mesma pasta

## Imports e Constantes

In [78]:
import csv
import json
from datetime import datetime, date

# ── Constantes ──────────────────────────────────────────────────────────────
ARQUIVO_CSV  = "transacoes.csv"
ARQUIVO_JSON = "relatorio.json"
LIMITE_SUSPEITO = 10_000.00
TIPOS_VALIDOS   = {"credito", "debito"}

print("✅ Definição de variáveis e constantes.")
print(f"   Arquivo de entrada : {ARQUIVO_CSV}")
print(f"   Arquivo de saída   : {ARQUIVO_JSON}")
print(f"   Limite suspeito    : R$ {LIMITE_SUSPEITO:,.2f}")

✅ Definição de variáveis e constantes.
   Arquivo de entrada : transacoes.csv
   Arquivo de saída   : relatorio.json
   Limite suspeito    : R$ 10,000.00


## Funções de Leitura e Validação

In [79]:
def validar_data(texto: str) -> datetime | None:
    """Converte texto 'AAAA-MM-DD' para datetime. Retorna None se inválido."""
    try:
        return datetime.strptime(texto.strip(), "%Y-%m-%d")
    except (ValueError, AttributeError):
        return None


def validar_valor(texto: str) -> float | None:
    """Converte texto para float positivo. Retorna None se inválido."""
    try:
        valor = float(texto.strip())
        return valor if valor > 0 else None
    except (ValueError, AttributeError):
        return None


def validar_transacao(linha: dict) -> dict | None:
    """
    Valida uma linha do CSV.
    Retorna o registro limpo como dict ou None se inválido.
    """
    # 1. id deve ser inteiro não-vazio
    try:
        id_transacao = int(linha.get("id", "").strip())
    except (ValueError, AttributeError):
        return None

    # 2. cliente_id não pode ser vazio
    cliente_id = linha.get("cliente_id", "").strip()
    if not cliente_id:
        return None

    # 3. data no formato AAAA-MM-DD
    data_dt = validar_data(linha.get("data", ""))
    if data_dt is None:
        return None

    # 4. tipo deve ser 'credito' ou 'debito'
    tipo = linha.get("tipo", "").strip().lower()
    if tipo not in TIPOS_VALIDOS:
        return None

    # 5. valor numérico e positivo
    valor = validar_valor(linha.get("valor", ""))
    if valor is None:
        return None

    return {
        "id":         id_transacao,
        "data":       data_dt,
        "mes":        data_dt.strftime("%Y-%m"),
        "cliente_id": cliente_id,
        "tipo":       tipo,
        "valor":      valor,
        "descricao":  linha.get("descricao", "").strip(),
        "categoria":  linha.get("categoria", "").strip(),
    }


def ler_transacoes(caminho: str) -> tuple[list[dict], int, int]:
    """
    Lê o CSV e retorna (transacoes_validas, total_lidas, total_invalidas).
    Informa caso não encontre o arquivo!
    """
    transacoes_validas = []
    total_lidas   = 0
    total_invalidas = 0

    try:
        with open(caminho, newline="", encoding="utf-8") as arquivo:
            leitor = csv.DictReader(arquivo)
            for linha in leitor:
                total_lidas += 1
                registro = validar_transacao(linha)
                if registro is not None:
                    transacoes_validas.append(registro)
                else:
                    total_invalidas += 1
    except FileNotFoundError:
        raise FileNotFoundError(
            f"❌ Arquivo '{caminho}' não encontrado. "
            "Crie o arquivo CSV antes de executar o notebook."
        )

    return transacoes_validas, total_lidas, total_invalidas


print("✅ Funções de leitura e validação definidas.")

✅ Funções de leitura e validação definidas.


## Geração de Métricas e Identificação de Suspeitas

In [80]:
def gerar_relatorio(
    transacoes: list[dict],
) -> tuple[dict, list[dict], datetime | None, datetime | None]:
    """
    Agrupa transações por mês e calcula métricas financeiras.

    Retorna:
        resumo_mensal  — dict com métricas por mês
        suspeitas      — lista de transações acima do LIMITE_SUSPEITO
        data_mais_antiga
        data_mais_recente
    """
    resumo: dict[str, dict] = {}
    suspeitas: list[dict]   = []

    datas = [t["data"] for t in transacoes]
    data_mais_antiga  = min(datas) if datas else None
    data_mais_recente = max(datas) if datas else None

    for t in transacoes:
        mes   = t["mes"]
        valor = t["valor"]
        tipo  = t["tipo"]

        # Inicializa o mês se ainda não existir
        if mes not in resumo:
            resumo[mes] = {
                "quantidade":    0,
                "total_credito": 0.0,
                "total_debito":  0.0,
                "_valores":      [],   # auxiliar para média / min / max
            }

        resumo[mes]["quantidade"]   += 1
        resumo[mes]["_valores"].append(valor)

        if tipo == "credito":
            resumo[mes]["total_credito"] += valor
        else:
            resumo[mes]["total_debito"]  += valor

        # Registra transação suspeita
        if valor > LIMITE_SUSPEITO:
            suspeitas.append(t)

    # Finaliza métricas derivadas
    for mes, dados in resumo.items():
        valores = dados.pop("_valores")          # remove campo auxiliar
        dados["saldo"]         = round(dados["total_credito"] - dados["total_debito"], 2)
        dados["media"]         = round(sum(valores) / len(valores), 2) if valores else 0.0
        dados["maior_valor"]   = max(valores) if valores else 0.0
        dados["menor_valor"]   = min(valores) if valores else 0.0
        dados["total_credito"] = round(dados["total_credito"], 2)
        dados["total_debito"]  = round(dados["total_debito"],  2)

    # Ordena os meses cronologicamente
    resumo = dict(sorted(resumo.items()))

    return resumo, suspeitas, data_mais_antiga, data_mais_recente


print("✅ Função gerar_relatorio definida.")

✅ Função gerar_relatorio definida.


## Exibição formatada no Terminal

In [81]:
def _brl(valor: float) -> str:
    """Formata float para o padrão brasileiro: R$ 1.234,56"""
    return f"R$ {valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def exibir_relatorio(
    resumo_mensal:   dict,
    suspeitas:       list[dict],
    total_validas:   int,
    total_invalidas: int,
    data_antiga:     datetime | None,
    data_recente:    datetime | None,
) -> None:
    """Imprime o relatório formatado no terminal."""

    SEPARADOR = "=" * 45

    # ── Cabeçalho ────────────────────────────────────────────────────────────
    print(f"\n{SEPARADOR}")
    print("    RELATÓRIO CLEARBANK")
    print(SEPARADOR)

    # Período
    if data_antiga and data_recente:
        fmt = "%d/%m/%Y"
        print(f"  Período  : {data_antiga.strftime(fmt)} → {data_recente.strftime(fmt)}")
        delta = (data_recente - data_antiga).days
        print(f"  Duração  : {delta} dias")

    print(f"  Válidas  : {total_validas}  |  Inválidas: {total_invalidas}")
    print(SEPARADOR)

    # ── Resumo mensal ────────────────────────────────────────────────────────
    print("\n===== RELATÓRIO MENSAL =====")
    for mes, d in resumo_mensal.items():
        print(f"\nMês: {mes}")
        print(f"  Transações   : {d['quantidade']}")
        print(f"  Total crédito: {_brl(d['total_credito'])}")
        print(f"  Total débito : {_brl(d['total_debito'])}")
        print(f"  Saldo        : {_brl(d['saldo'])}")
        print(f"  Média        : {_brl(d['media'])}")
        print(f"  Maior valor  : {_brl(d['maior_valor'])}")
        print(f"  Menor valor  : {_brl(d['menor_valor'])}")

    # ── Transações suspeitas ─────────────────────────────────────────────────
    print(f"\n{SEPARADOR}")
    print("===== TRANSAÇÕES SUSPEITAS =====")
    if suspeitas:
        for t in suspeitas:
            print(
                f"  ID: {t['id']:>4} | Cliente: {t['cliente_id']:<8} "
                f"| Data: {t['data'].strftime('%Y-%m-%d')} "
                f"| Valor: {_brl(t['valor'])}"
            )
    else:
        print("  Nenhuma transação suspeita encontrada.")

    print(f"{SEPARADOR}\n")


print("✅ Funções de exibição definidas.")

✅ Funções de exibição definidas.


## Exportação JSON

In [82]:
def salvar_json(
    caminho:         str,
    resumo_mensal:   dict,
    suspeitas:       list[dict],
    total_validas:   int,
    total_invalidas: int,
) -> None:
    """
    Relatório completo -> arquivo JSON.
    """
    # Prepara lista de suspeitas serializável
    suspeitas_serial = [
        {
            "id":         t["id"],
            "data":       t["data"].strftime("%Y-%m-%d"),
            "cliente_id": t["cliente_id"],
            "tipo":       t["tipo"],
            "valor":      t["valor"],
            "descricao":  t["descricao"],
            "categoria":  t["categoria"],
        }
        for t in suspeitas
    ]

    payload = {
        "gerado_em":                   date.today().isoformat(),
        "total_transacoes_validas":    total_validas,
        "total_transacoes_invalidas":  total_invalidas,
        "limite_suspeito":             LIMITE_SUSPEITO,
        "resumo_mensal":               resumo_mensal,
        "transacoes_suspeitas":        suspeitas_serial,
    }

    try:
        with open(caminho, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        print(f"✅ Relatório salvo em '{caminho}'")
    except OSError as e:
        print(f"❌ Erro ao salvar JSON: {e}")


print("✅ Função salvar_json definida.")

✅ Função salvar_json definida.


## Análise Exploratória de Datas

In [83]:
# Lê o arquivo para demonstrar o cálculo de período antes da execução principal
try:
    _transacoes_demo, _total_demo, _inv_demo = ler_transacoes(ARQUIVO_CSV)
    _datas = [t["data"] for t in _transacoes_demo]

    if _datas:
        _antiga  = min(_datas)
        _recente = max(_datas)
        _delta   = (_recente - _antiga).days

        print("📅 Análise de Datas")
        print(f"   Transação mais antiga : {_antiga.strftime('%d/%m/%Y')}")
        print(f"   Transação mais recente: {_recente.strftime('%d/%m/%Y')}")
        print(f"   Período abrangido      : {_delta} dias")
        print(f"   Meses distintos        : {len({t['mes'] for t in _transacoes_demo})}")
        print(f"   Linhas válidas         : {len(_transacoes_demo)}")
        print(f"   Linhas inválidas       : {_inv_demo}")
    else:
        print("Nenhuma transação válida encontrada.")

except FileNotFoundError as e:
    print(e)

📅 Análise de Datas
   Transação mais antiga : 05/01/2026
   Transação mais recente: 28/04/2026
   Período abrangido      : 113 dias
   Meses distintos        : 4
   Linhas válidas         : 20
   Linhas inválidas       : 5


---
## 🚀 Célula de Execução Principal

> Execute esta célula **após** todas as anteriores. Ela integra todas as funções e gera o relatório completo.

In [85]:
def main():
    """Programa principal!"""

    # Leitura e validação
    try:
        transacoes, total_lidas, total_invalidas = ler_transacoes(ARQUIVO_CSV)
    except FileNotFoundError as e:
        print(e)
        return

    total_validas = len(transacoes)

    print("\n===== RESUMO DA LIMPEZA =====")
    print(f"  Total de linhas lidas: {total_lidas}")
    print(f"  Linhas válidas       : {total_validas}")
    print(f"  Linhas inválidas     : {total_invalidas}")

    if not transacoes:
        print("⚠️  Nenhuma transação válida para processar.")
        return

    # Geração do relatório mensal
    resumo_mensal, suspeitas, data_antiga, data_recente = gerar_relatorio(transacoes)

    # Exibição formatada
    exibir_relatorio(
        resumo_mensal,
        suspeitas,
        total_validas,
        total_invalidas,
        data_antiga,
        data_recente,
    )

    # Exportação arquivo JSON
    salvar_json(
        ARQUIVO_JSON,
        resumo_mensal,
        suspeitas,
        total_validas,
        total_invalidas,
    )


# Execução
main()


===== RESUMO DA LIMPEZA =====
  Total de linhas lidas: 25
  Linhas válidas       : 20
  Linhas inválidas     : 5

    RELATÓRIO CLEARBANK
  Período  : 05/01/2026 → 28/04/2026
  Duração  : 113 dias
  Válidas  : 20  |  Inválidas: 5

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações   : 5
  Total crédito: R$ 9.500,00
  Total débito : R$ 430,50
  Saldo        : R$ 9.069,50
  Média        : R$ 1.986,10
  Maior valor  : R$ 4.800,00
  Menor valor  : R$ 180,50

Mês: 2026-02
  Transações   : 5
  Total crédito: R$ 18.500,00
  Total débito : R$ 859,90
  Saldo        : R$ 17.640,10
  Média        : R$ 3.871,98
  Maior valor  : R$ 15.000,00
  Menor valor  : R$ 89,90

Mês: 2026-03
  Transações   : 6
  Total crédito: R$ 18.800,00
  Total débito : R$ 1.374,90
  Saldo        : R$ 17.425,10
  Média        : R$ 3.362,48
  Maior valor  : R$ 12.500,00
  Menor valor  : R$ 75,00

Mês: 2026-04
  Transações   : 4
  Total crédito: R$ 9.500,00
  Total débito : R$ 840,00
  Saldo        : R$ 8.660,00
  Médi